# 0.12 · 工程化基础 / Engineering Basics

> **课程定位 / Where this fits**
> 第 12 课 = **Part 0 的收官节**。
> Lesson 12 = **the capstone of Part 0**.
>
> 前 11 节我们打通了 **数据 + 工具 + 数学**。这一节讲**怎么把它们组装成"可被同事/招聘官/未来的你"打开就能跑的项目**——Git、虚拟环境、依赖管理、Jupyter、VSCode、可复现性、代码质量。
> The previous 11 lessons covered data + tools + math. This one shows **how to package them into a project anyone can open and run** — Git, venvs, deps, Jupyter, VSCode, reproducibility, code quality.

> 💡 **为什么单独成节 / Why a dedicated lesson**
> 工业界的 DS 数据科学家**至少 30% 时间花在工程**而不是建模。Junior 和 Senior 的差距常常在这里。**写得起跑得起的代码 >> 模型多花哨**。
> Industry DS spend ≥30% of their time on engineering. The Junior-vs-Senior gap often lives here. **Reproducible code beats fancy models.**

> 📐 **符号约定 / Notation**
> 本节几乎没数学，主要是命令行 + 文件结构。
> No math here — just commands + file structure.

> 💡 **面试相关 / Interview-relevant**
> - "解释你的 DS 项目结构" ★★★★★（system design 必问）
> - "怎么保证模型可复现" ★★★★（MLOps 入门）
> - "你怎么管理依赖" ★★★
> - "Git workflow 用什么" ★★★
>
> Most-asked: project structure, reproducibility, dep management, Git workflow.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 用 **`cookiecutter-data-science`** 风格的项目骨架开新 DS 项目。
   Spin up a new DS project using the `cookiecutter-data-science` layout.
2. 写出一份**完整的 `.gitignore`**，避免把数据/模型/秘密推到 GitHub。
   Write a complete `.gitignore` so you never leak data / models / secrets.
3. 比较 **`venv` / `conda` / `uv` / `pixi`** 四种环境管理工具，选适合你的。
   Compare the four env managers and pick one.
4. 区分 **`requirements.txt` / `pyproject.toml` / lockfile** 各自用途。
   Tell apart `requirements.txt`, `pyproject.toml`, and lock files.
5. 用 **`autoreload`、命名规范、kernel 选择**让 Jupyter 不再变成"屎山"。
   Use autoreload, naming conventions, and kernel selection to keep notebooks sane.
6. 配 **VSCode + ruff + mypy + pre-commit**——团队代码质量自动护栏。
   Wire up VSCode + ruff + mypy + pre-commit — automated quality guardrails.
7. 写出**严格可复现**的训练代码（种子 + 配置 + 环境哈希）。
   Write strictly reproducible training code (seeds + config + env hash).
8. 把上面所有东西**串成一个一键启动的项目模板**。
   Tie everything into a one-command project template.

---

## 目录 / Table of Contents

1. [项目结构 / Project Structure](#1)
2. [Git for DS ⭐](#2)
3. [虚拟环境 / Virtual Environments](#3)
4. [依赖管理 / Dependency Management](#4)
5. [Jupyter 最佳实践 / Jupyter Best Practices ⭐](#5)
6. [VSCode 配置 / VSCode Setup](#6)
7. [代码质量自动化 / Code Quality Automation](#7)
8. [可复现性 ⭐ / Reproducibility](#8)
9. [日志而非 print / Logging not `print`](#9)
10. [项目模板汇总 / Project Template Summary](#10)
11. [Part 0 总结 / Part 0 Wrap-up](#11)


<a id="1"></a>
## 1. 项目结构 / Project Structure

### 1.1 业界标准：cookiecutter-data-science / Industry standard

[`cookiecutter-data-science`](https://drivendata.github.io/cookiecutter-data-science/) 是 Driven Data 开源的 DS 项目骨架，**几乎所有大厂内部模板**都从它演化。
A widely-used reference template; most big-tech internal templates derive from it.

```text
my_ds_project/
├── README.md                ← 项目目标、怎么跑、关键结论
├── pyproject.toml           ← 依赖、配置工具（ruff/mypy）、metadata
├── uv.lock  / requirements.txt  ← 锁定具体版本
├── .gitignore
├── .pre-commit-config.yaml
│
├── data/
│   ├── raw/                 ← ⚠ 永远不修改的原始数据 / immutable raw data
│   ├── interim/             ← 中间清洗后 / intermediate cleaned
│   ├── processed/           ← 可直接喂模型 / model-ready
│   └── external/            ← 外部源（API/SQL dump）/ from outside
│
├── notebooks/               ← 探索用 .ipynb
│   ├── 01_eda.ipynb
│   ├── 02_features.ipynb
│   └── 03_model.ipynb
│
├── src/
│   └── my_project/          ← 真正的 Python 包 / actual Python package
│       ├── __init__.py
│       ├── config.py        ← 配置 (paths, hparams)
│       ├── data.py          ← 数据加载/清洗
│       ├── features.py      ← 特征工程
│       ├── models.py        ← 模型定义
│       ├── train.py         ← 训练入口
│       └── evaluate.py      ← 评估
│
├── tests/                   ← pytest 单元测试
│   ├── test_data.py
│   └── test_models.py
│
├── scripts/                 ← CLI 入口（`python scripts/train.py`）
│   └── train_main.py
│
├── reports/
│   ├── figures/             ← 论文/PPT 用的图
│   └── final_report.md
│
└── models/                  ← 训练好的模型文件 (gitignore'd)
    └── README.md
```

### 1.2 关键三原则 / Three guiding principles

1. **`data/raw/` 永不修改** —— 任何派生数据写到 `interim/` 或 `processed/`，可重生成。
   `data/raw/` is **immutable**; everything else regenerable.
2. **`notebooks/` 只做探索** —— 真正的逻辑提到 `src/` 里，让 notebook 只调用函数。
   Notebooks for exploration; real logic in `src/`.
3. **CLI 入口在 `scripts/`** —— 这样你可以 `python scripts/train.py --config conf.yaml`。
   CLI lives in `scripts/`.

> 💡 **面试一句话答 / One-line interview**
> "Project layout follows cookiecutter-data-science: immutable raw data, notebooks for exploration only, library code in `src/`, CLI in `scripts/`, tests in `tests/`."


<a id="2"></a>
## 2. Git for DS ⭐

### 2.1 最小可用流程 / Minimum workable workflow

```bash
git init
git checkout -b feature/eda          # 总是在分支上工作 / always branch
# ... edit, run, commit ...
git add notebooks/01_eda.ipynb src/my_project/data.py
git commit -m "EDA: confirm class imbalance, age missing rate 20%"
git push -u origin feature/eda
# 然后开 PR / open PR
```

### 2.2 ⭐ `.gitignore` for DS

**永远不能进 git** 的东西：
Things that **must never** enter git:

```gitignore
# === Python ===
__pycache__/
*.py[cod]
.venv/
venv/
env/
.python-version

# === Jupyter ===
.ipynb_checkpoints/
*-checkpoint.ipynb

# === IDE / OS ===
.vscode/
.idea/
.DS_Store
Thumbs.db

# === Data (the BIG one) ===
data/raw/
data/interim/
data/processed/
data/external/
# 但保留 README 说明数据来源 / but keep README explaining sources
!data/**/README.md
!data/sample_*.csv          # 小示例可以保留 / tiny samples ok

# === Models ===
models/
*.pkl
*.joblib
*.pt
*.pth
*.onnx
*.ckpt
mlruns/                     ← MLflow
wandb/                      ← Weights & Biases
lightning_logs/
catboost_info/

# === Secrets ⚠⚠⚠ ===
.env
*.key
*.pem
credentials.json
secrets/
config/local.yaml

# === Logs ===
*.log
tmp/
.cache/
```

### 2.3 大文件怎么办 / Big files

不要把 GB 级文件推到 git——会让 clone 慢、push 失败。
Don't push GB files to git.

| 方案 / Option | 适用 / When |
|---|---|
| **Git LFS** | 模型文件（< 几个 GB），团队都装得起 LFS |
| **DVC** (Data Version Control) | 数据版本管理标配，跟 git tag 联动 |
| **S3 / GCS** + 在 README 写 `gsutil cp ...` | 最简单，无 LFS 配额烦 |

### 2.4 一句话 commit message 规范 / Commit style

类型: 一句话总结
- `feat:` 新功能
- `fix:` bug 修复
- `refactor:` 重构（不改行为）
- `test:` 测试
- `docs:` 文档
- `chore:` 杂项（依赖更新等）

例：
```
feat(eda): add survival × class × sex heatmap
fix(features): handle NaN in family_size derivation
refactor(train): extract config parsing
```

### 2.5 常用命令速查 / Cheat sheet


In [ ]:
# 一些可以本地跑的 git 演示 / Demo git commands
import subprocess, tempfile, os
from pathlib import Path

_orig_cwd = os.getcwd()

# 创建一个临时仓库演示 / create a temp repo to demo
with tempfile.TemporaryDirectory() as tmp:
    os.chdir(tmp)

    # init
    subprocess.run(["git", "init", "-b", "main", "-q"], check=True)

    # 配 user（CI/临时环境必须）/ configure (CI/temp env needs this)
    subprocess.run(["git", "config", "user.email", "demo@x.com"], check=True)
    subprocess.run(["git", "config", "user.name", "Demo"], check=True)

    # 写一个文件，提交 / write a file, commit
    Path("hello.py").write_text("print('hi')\n")
    subprocess.run(["git", "add", "hello.py"], check=True)
    subprocess.run(["git", "commit", "-m", "first commit", "-q"], check=True)

    # 显示 log
    out = subprocess.run(["git", "log", "--oneline"], capture_output=True, text=True)
    print("git log:")
    print(out.stdout)

    # 切分支 / branch
    subprocess.run(["git", "checkout", "-b", "feature/test", "-q"], check=True)
    Path("hello.py").write_text("print('hi from feature branch')\n")
    subprocess.run(["git", "add", "hello.py"], check=True)
    subprocess.run(["git", "commit", "-m", "update msg", "-q"], check=True)

    # 看分支 / branches
    out = subprocess.run(["git", "branch"], capture_output=True, text=True)
    print("\nbranches:")
    print(out.stdout)

# 恢复原 cwd（temp dir 已被删）/ Restore cwd (temp dir is gone)
os.chdir(_orig_cwd)


<a id="3"></a>
## 3. 虚拟环境 / Virtual Environments

**为什么必须用 / Why mandatory**：你 5 个项目，每个对 numpy / torch 版本要求不同。装到全局 Python = 灾难。
You'll have 5 projects with different version constraints; installing to system Python = disaster.

### 四个主流工具 / Four mainstream tools

| 工具 / Tool | 速度 | 跨语言（带 R / Julia / C 库）| 学习曲线 | 现状 |
|---|---|---|---|---|
| **`venv`** | 中 | ❌ Python only | 极低 / minimal | Python 自带，最稳妥 |
| **`conda` / mamba** | 慢 | ✅ 强 | 中等 | DS 老牌、笨重但功能强 |
| **`uv`** ⭐ | **极快**（Rust 写的）| ❌ Python only | 低 | **2024 起新王者**，速度 10-100× |
| **`pixi`** | 快 | ✅ conda-like | 中 | conda 的现代替代 |

### 推荐 / Recommendation

| 你是 / You are | 用 / Use |
|---|---|
| DS 新手 / 学生 | `uv` —— 一个命令搞定所有 |
| 需要非 Python 库（CUDA、R）| `conda/mamba` 或 `pixi` |
| 团队约定老传统 | 跟着团队 |
| 写论文要"零安装麻烦"| `uv` + `uv.lock` |

### 3.1 `venv` 用法 / `venv` usage

```bash
# 创建 + 激活 / create + activate
python3 -m venv .venv
source .venv/bin/activate         # macOS / Linux
.venv\\Scripts\\activate            # Windows

# 装包 / install
pip install numpy pandas

# 退出 / deactivate
deactivate
```

### 3.2 `conda` / `mamba` 用法

```bash
# 用 mamba 替换 conda 即可，10-100x 快 / mamba is a drop-in faster replacement
conda create -n my_project python=3.11
conda activate my_project
conda install -c conda-forge numpy pandas scikit-learn
```

### 3.3 `uv` 用法 ⭐

```bash
# 一次性安装 / one-time install
curl -LsSf https://astral.sh/uv/install.sh | sh

# 在项目里
uv init                                    # 创建 pyproject.toml
uv add numpy pandas scikit-learn           # 加依赖 + 自动 lock
uv add --dev pytest ruff mypy              # dev 依赖
uv run python scripts/train.py             # 自动激活环境跑
uv sync                                    # 严格按 lock 文件安装（CI/同事拉代码）
```

**`uv` 最大卖点**：
- pip install 100 MB 的 numpy 包 → 几秒，不是几分钟
- 自动管 `.python-version` 和 lock 文件
- 不再需要 `venv activate` —— `uv run` 直接搞定


<a id="4"></a>
## 4. 依赖管理 / Dependency Management

### 三种文件 / Three files

| 文件 / File | 是什么 | 适用 |
|---|---|---|
| **`requirements.txt`** | 平铺包列表 | 最古老、最简单。可有版本范围 (`>=`) 或固定 (`==`) |
| **`pyproject.toml`** ⭐ | 现代 Python 项目元信息 | PEP 621 标准，工具配置也写这里 |
| **`uv.lock` / `poetry.lock`** | 锁文件 | 记录**所有间接依赖**的精确版本 + 哈希 |

### 4.1 `requirements.txt` 的两种用法 / Two styles

```text
# 风格 A：松散，开发用 / loose, for dev
numpy>=1.26
pandas>=2.2
scikit-learn>=1.5

# 风格 B：严格，部署用 / pinned, for prod
numpy==1.26.4
pandas==2.2.2
scikit-learn==1.5.0
```

> ⚠ 直接 `pip freeze > requirements.txt` 会把**整个环境**（含 transitive deps）写进去，糙但有效。
> `pip freeze` dumps the whole env (incl. transitive deps) — crude but works.

### 4.2 现代做法：`pyproject.toml`

```toml
[project]
name = "my_ds_project"
version = "0.1.0"
description = "Predict customer churn"
requires-python = ">=3.11"
dependencies = [
    "numpy>=1.26",
    "pandas>=2.2",
    "scikit-learn>=1.5",
]

[project.optional-dependencies]
dev = ["pytest>=8", "ruff>=0.5", "mypy>=1.10", "pre-commit>=3"]

[tool.ruff]
line-length = 100
select = ["E", "F", "I"]

[tool.mypy]
strict = true

[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"
```

**优势 / Wins**：
- 一个文件管：依赖 + 工具配置 + 元数据 + 入口脚本
- `pip install .` 把项目装成包，让 `from my_project import ...` 工作
- 现代工具（uv / hatch / pdm / poetry）都吃这个

### 4.3 Lock files 为什么重要 / Why lock files matter

`pyproject.toml` 说"numpy>=1.26"，但**今天**装到 1.26.4，**下个月**可能装到 1.27.2 —— **可复现性挂掉**。
The constraint says ">=1.26", but today you get 1.26.4, next month 1.27.2 — reproducibility broken.

锁文件**冻结所有依赖的精确版本 + 哈希**：
Lock files freeze every dep:

```text
# uv.lock 节选
[[package]]
name = "numpy"
version = "1.26.4"
source = { registry = "https://pypi.org/simple" }
sdist = { url = "...", hash = "sha256:abc..." }
```

**规则 / Rule**：`pyproject.toml` 是 source of truth；`uv.lock` **提交到 git**；其他人 `uv sync` 就能精确复现你的环境。
Commit the lock file → others get the exact same environment via `uv sync`.


<a id="5"></a>
## 5. Jupyter 最佳实践 / Jupyter Best Practices ⭐

Notebook 是 DS 的"双刃剑"——快速探索神器，但**写不好就是技术债的源头**。
Notebooks are a double-edged sword — great for exploration, terrible if abused.

### 5.1 几条铁律 / Hard rules

1. **kernel 必须用项目的 venv** —— 不要用"全局 Python 3"
2. **代码逻辑提到 `.py`，notebook 只调用** —— 用 `%load_ext autoreload`
3. **从上到下能跑通** —— 提交前 `Restart & Run All`
4. **不要 commit 大的输出 cell**（图片、表格）—— 用 `nbstripout` 自动清
5. **命名加序号** —— `01_eda.ipynb`、`02_features.ipynb`...
6. **大数据集开头**就**采样**到几千行做开发，最后再 full run
7. **不要藏全局变量**：cell B 用了 cell A 定义的东西，B 应该能独立 re-run

### 5.2 `autoreload`：开发时神器 / Dev superpower

放在 notebook 顶部：
```python
%load_ext autoreload
%autoreload 2
```

之后你改 `src/my_project/features.py`，**notebook 不用重启**自动重新 import。**写 notebook + 库代码混合开发**必备。
After this, edits to `.py` files are picked up without restarting the kernel.

### 5.3 把 venv 注册成 Jupyter kernel

新装的 venv 默认 Jupyter 看不到。注册一下：
A new venv isn't visible to Jupyter by default — register it:

```bash
source .venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=my_project --display-name="my_project"
```

然后 Jupyter 顶部 "Kernel" 菜单里就能选了。
Now it appears in Jupyter's kernel menu.

### 5.4 一份示范 notebook 头部 / Sample notebook header


In [ ]:
# === 示范：一个干净 notebook 应该这样开头 ===
# Sample clean notebook header (this cell is for demo)

# 1) 让 .py 模块改完自动重载 / auto-reload edited .py files
#    在真实 notebook 顶部加这两行（这里注释掉，因为是普通 .py 单元）
# %load_ext autoreload
# %autoreload 2

# 2) 标准库 + 第三方
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 3) 抑制无关警告 / Quiet common annoyances
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# 4) 输出/绘图风格 / Output + plot style
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (8, 5)

# 5) 全局种子 / Global seed
SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# 6) 路径常量（不写硬编码！）/ Path constants (no hardcoded paths!)
PROJECT_ROOT = Path.cwd()       # 假设 notebook 在 notebooks/ 下，会用 .parent
DATA_DIR = PROJECT_ROOT / "data"

# 7) 环境信息（出问题时方便定位）/ Environment info (for debugging)
print(f"Python    : {sys.version.split()[0]}")
print(f"numpy     : {np.__version__}")
print(f"pandas    : {pd.__version__}")
print(f"seed      : {SEED}")


### 5.5 nbstripout：避免 diff 噪声

```bash
pip install nbstripout
nbstripout --install              # 在当前 repo 装钩子 / install hook in repo
```

之后 `git add notebook.ipynb` 自动剥掉输出 cell，commit 时只看代码变化。**团队协作必备**。
Strips output cells on commit; essential for team collaboration.


<a id="6"></a>
## 6. VSCode 配置 / VSCode Setup

VSCode 是目前 DS 圈占有率最高的 IDE，完全免费。
The most popular IDE in DS today, fully free.

### 6.1 必装扩展 / Must-have extensions

| 扩展 / Extension | 作用 |
|---|---|
| **Python** (ms-python.python) | 基础 Python 支持 + debugger |
| **Pylance** | 类型检查、自动补全 |
| **Jupyter** | 在 VSCode 里直接跑 .ipynb |
| **Ruff** (charliermarsh.ruff) | 一键 lint + format |
| **Git Graph** | 可视化 git 历史 |
| **GitHub Pull Requests** | 在编辑器里管 PR |
| **Remote - SSH** | 远程开发（连 GPU 服务器必备）|
| **Even Better TOML** | `pyproject.toml` 语法高亮 |
| **YAML** | 配置文件 |

### 6.2 推荐 `settings.json` 片段 / Recommended settings

放到 `.vscode/settings.json`（仓库级，可以 commit）：
Workspace settings go in `.vscode/settings.json`:

```jsonc
{
  // 自动用项目里的 venv / Auto-select project venv
  "python.defaultInterpreterPath": "${workspaceFolder}/.venv/bin/python",

  // 保存时自动 lint + format / Lint + format on save
  "editor.formatOnSave": true,
  "editor.codeActionsOnSave": {
    "source.organizeImports": "explicit",
    "source.fixAll.ruff": "explicit"
  },
  "[python]": {
    "editor.defaultFormatter": "charliermarsh.ruff"
  },

  // pytest 自动发现 / Pytest discovery
  "python.testing.pytestEnabled": true,
  "python.testing.pytestArgs": ["tests"],

  // 跑 notebook 用项目 venv / Notebooks use project venv
  "jupyter.notebookFileRoot": "${workspaceFolder}",

  // 让 Pylance 看到 src/ 包
  "python.analysis.extraPaths": ["./src"],

  // 不要 commit 这俩 / don't commit
  "files.exclude": {
    "**/__pycache__": true,
    "**/.ipynb_checkpoints": true
  }
}
```

### 6.3 三个超有用快捷键 / Three killer shortcuts

| 操作 / Action | macOS | Windows |
|---|---|---|
| Command Palette（万能入口）| `⌘⇧P` | `Ctrl+Shift+P` |
| 在 Notebook 里运行 cell 并跳到下一个 | `⇧↩` | `Shift+Enter` |
| Go to Definition | `F12` | `F12` |
| Find in all files | `⌘⇧F` | `Ctrl+Shift+F` |
| 重命名变量（所有引用一起改）| `F2` | `F2` |

### 6.4 远程开发 / Remote development

DS 经常要在 GPU 服务器跑训练。VSCode Remote-SSH 让你**本地编辑、远程执行、零延迟**：

```bash
# 本地 ~/.ssh/config
Host my-gpu-box
  HostName gpu.lab.example.com
  User zachary
  IdentityFile ~/.ssh/id_ed25519
```

然后 VSCode 左下角点 "Connect to Host" → 选 `my-gpu-box` → 就像本地一样使用。
Then VSCode connects and the dev experience feels local.


<a id="7"></a>
## 7. 代码质量自动化 / Code Quality Automation

### 7.1 工具栈 / The stack

| 工具 / Tool | 作用 |
|---|---|
| **`ruff`** ⭐ | **极快**的 linter + formatter（Rust 写的，替代 black + flake8 + isort + ...）|
| **`mypy`** | 静态类型检查（catch bugs before runtime） |
| **`pytest`** | 单元测试 |
| **`pre-commit`** ⭐ | git commit 前**自动跑**上面所有工具 |

### 7.2 一个最小 `pyproject.toml` 配置

```toml
[tool.ruff]
line-length = 100
target-version = "py311"
src = ["src"]

[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP", "RUF"]
# E = pycodestyle errors
# F = pyflakes
# I = isort (import order)
# B = bugbear (common bugs)
# UP = pyupgrade (modernize syntax)
# RUF = ruff-specific

[tool.mypy]
python_version = "3.11"
strict = true
ignore_missing_imports = true   # 不强求第三方库类型 / don't fail on stubs

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"
```

### 7.3 `pre-commit`：质量护栏 / Quality gates

`.pre-commit-config.yaml`:

```yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.5.0
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format

  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.6.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-added-large-files
        args: ['--maxkb=500']     ← 防止误把数据/模型 add 进来

  - repo: https://github.com/kynan/nbstripout
    rev: 0.7.1
    hooks:
      - id: nbstripout

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.10.0
    hooks:
      - id: mypy
        additional_dependencies: [numpy, pandas]
```

激活 / Enable:
```bash
pip install pre-commit
pre-commit install            # 装到 .git/hooks/
pre-commit run --all-files    # 第一次手动跑一遍
```

之后**每次 `git commit` 都会自动检查**——不通过就不让 commit。"忘了 format"再也不会发生。
After this every `git commit` runs the checks automatically — "forgot to format" never happens again.


<a id="8"></a>
## 8. 可复现性 ⭐ / Reproducibility

**面试 ★★★★** 高频题。一个模型今天准确率 85%，明天 84.7%，**为什么**？
Interview classic: why does the model give 85% today and 84.7% tomorrow?

### 8.1 必须固定的"随机源" / Sources of randomness to control

| 来源 / Source | 怎么固定 / How to seed |
|---|---|
| `random` (Python stdlib) | `random.seed(42)` |
| `numpy` | `np.random.default_rng(42)` ⭐ |
| `torch` CPU | `torch.manual_seed(42)` |
| `torch` GPU | `torch.cuda.manual_seed_all(42)` |
| `torch` cuDNN backend | `torch.backends.cudnn.deterministic = True` |
| `tensorflow` | `tf.random.set_seed(42)` |
| sklearn 的部分模型 | 传 `random_state=42` 给构造函数 |
| 数据切分 | `train_test_split(..., random_state=42)` |
| Python hash 随机化 | 设环境变量 `PYTHONHASHSEED=42` |

> 💡 **完全 deterministic 在 GPU 上几乎不可能** — 浮点累计顺序受并行调度影响。能做到 "run-to-run 一致到 1e-5" 就算工程合格。
> Bit-exact reproducibility on GPU is nearly impossible; matching to ~1e-5 across runs is the realistic bar.

### 8.2 一个"可复现"的训练入口模板 / Reproducible training entry-point


In [ ]:
# 标准可复现训练入口模板 / Canonical reproducible training entry-point
from dataclasses import dataclass, asdict, field
import hashlib
import json
import os
import random
import platform
import sys
from pathlib import Path

import numpy as np

# =========================================================
# 1) 配置（永远用 dataclass，从不散落 magic numbers）
# 1) Config as dataclass — no scattered magic numbers
# =========================================================
@dataclass
class TrainConfig:
    seed: int = 42
    learning_rate: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    val_split: float = 0.2
    model: str = "logistic"
    extra: dict = field(default_factory=dict)

# =========================================================
# 2) 种子统一函数 / Seed everything in one call
# =========================================================
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

# =========================================================
# 3) 环境指纹（出 bug 时方便定位）
# 3) Env fingerprint (debugging gold)
# =========================================================
def env_fingerprint() -> dict:
    info = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
    }
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.cuda.is_available()
    except ImportError:
        pass
    try:
        import pandas as pd
        info["pandas"] = pd.__version__
    except ImportError:
        pass
    return info

# =========================================================
# 4) 把 (config + env) 哈希到 run_id —— 同样的 config 总给同样的 run_id
# 4) Hash (config + env) into a deterministic run_id
# =========================================================
def make_run_id(config: TrainConfig, env: dict) -> str:
    payload = json.dumps({"config": asdict(config), "env": env}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

# === 演示 / Demo ===
cfg = TrainConfig(seed=42, learning_rate=1e-3, epochs=5)
env = env_fingerprint()
set_seed(cfg.seed)

run_id = make_run_id(cfg, env)
print(f"config : {cfg}")
print(f"env    : {env}")
print(f"run_id : {run_id}")


**关键点 / Key**：
- 同样的 config + 同样的环境 → 同样的 `run_id`（哈希一致）
- 把 `run_id` 写进**所有产物**（模型文件名、log 目录、wandb run name）
- 出问题时一眼能查到："哪个 commit + 哪个 config 跑出来的"
- Write the `run_id` into every artifact (model filename, log dir, wandb run name) so you can always trace results back.


<a id="9"></a>
## 9. 日志而非 print / Logging not `print`

`print` 在脚本里没问题，但**真正训练时**你需要：
`print` works for scripts; for real training you need:

- 时间戳 / timestamps
- 不同级别（DEBUG/INFO/WARNING/ERROR）/ severity levels
- 输出到**文件 + 屏幕**两边 / file **and** console
- 在生产环境里能被 grep / ELK / CloudWatch 摄取

Python 标准库 `logging` 都满足，5 行配好。
Python's stdlib `logging` covers all of this in ~5 lines.


In [ ]:
# 一个生产可用的 logging 配置 / Production-ready logging setup
import logging
from pathlib import Path

def get_logger(name: str, log_file: Path | None = None) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:                # 避免重复添加 handler / avoid double-add
        return logger
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter(
        "%(asctime)s [%(levelname)s] %(name)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # 屏幕 / Console
    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # 文件 / File (optional)
    if log_file:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file)
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger

# === 用法 / Usage ===
log = get_logger("train", log_file=Path("/tmp/demo_train.log"))
log.info("Starting training with batch_size=32")
log.warning("Validation loss plateaued for 3 epochs")
log.error("OOM on batch 1250 — reducing batch size to 16")
log.info("Saved model to /tmp/model.pt")

print("\n--- log file content ---")
print(Path("/tmp/demo_train.log").read_text())


**为什么这是工业最佳实践 / Why this is the industry default**:
- 跑 24 小时训练时，看 `tail -f train.log` 比看 print buffer 安心多了
- 出问题时直接 `grep ERROR train.log` 定位
- 多个训练并行 → 各写各的 log 文件，互不串


<a id="10"></a>
## 10. 项目模板汇总 / Project Template Summary

把所有东西串起来：**一个新的 DS 项目从 0 到 ready** 大概这样：
A new DS project from scratch:

```bash
# 1. 建目录 + git
mkdir my_project && cd my_project
git init -b main
mkdir -p src/my_project tests notebooks scripts data/{raw,interim,processed} reports/figures models

# 2. 用 uv 装环境 + 依赖
uv init --no-readme
uv add numpy pandas scikit-learn matplotlib seaborn plotly polars
uv add --dev pytest ruff mypy pre-commit nbstripout ipykernel

# 3. 项目骨架文件 / scaffold files
touch README.md src/my_project/__init__.py
touch src/my_project/{config,data,features,models,train,evaluate}.py
touch .gitignore .pre-commit-config.yaml

# 4. 启用 pre-commit hooks
uv run pre-commit install

# 5. 注册 jupyter kernel
uv run python -m ipykernel install --user --name=my_project --display-name=my_project

# 6. 第一次提交
git add .
git commit -m "chore: initial project scaffold"
```

### 一份"开箱即用"README 模板 / README template

```markdown
# Project Name

> 一句话项目目的。/ One-sentence project purpose.

## Setup

\`\`\`bash
uv sync                       # install exact deps from uv.lock
uv run pre-commit install     # enable quality hooks
\`\`\`

## Reproduce

\`\`\`bash
uv run python scripts/train.py --config configs/baseline.yaml
\`\`\`

## Project Structure
- `src/`        library code
- `notebooks/`  exploratory analysis
- `scripts/`    CLI entry-points
- `tests/`      pytest

## Results
| Metric | Value |
|---|---|
| ROC-AUC (val) | 0.87 |
| F1 (val)      | 0.79 |

## Caveats
- Class imbalance ratio = 1:8 — use `class_weight='balanced'`
- `deck` column dropped (77% missing)
```

### 💡 工业现实 / Industry reality

90% 的招聘官 / co-worker **第一眼看 `README.md`**。一份糟糕的 README = 一个被低估的 project。
First impression = `README.md`. Bad README = underestimated project.


<a id="11"></a>
## 11. Part 0 总结 / Part 0 Wrap-up 🎉

**恭喜——你完成了 Part 0 基础准备！**
**Congrats — Part 0 done!**

### 已掌握 / What you've mastered

| # | 模块 / Module | 核心 / Core |
|---|---|---|
| 0.1 | Python for DS | 容器、推导式、装饰器、OOP、`dataclass` |
| 0.2 | NumPy | ndarray、广播、`einsum`、SVD-ready 线代 |
| 0.3 | Pandas | `.loc/.iloc`、`groupby`、`merge`、`pivot` |
| 0.4 | Polars | Expression API、lazy、5-10× pandas 速度 |
| 0.5 | Matplotlib & Seaborn | Figure/Axes 模型、15+ 图表 |
| 0.6 | Plotly | 交互、动画、地图、HTML 导出 |
| 0.7 | Linear Algebra | 变换、特征值、谱定理、SVD、手写 PCA |
| 0.8 | Calculus | 梯度、链式法则、Hessian、autograd |
| 0.9 | Probability | Bayes、10 大分布、CLT、MLE、Naive Bayes |
| 0.10 | Optimization | 凸性、KKT、Newton、L-BFGS、Adam 家族 |
| 0.11 | Information Theory | 熵、KL、CE、互信息、决策树信息增益 |
| 0.12 | Engineering | 项目结构、git、venv、Jupyter、可复现 |

### 已搭出的工具箱 / Your stable toolkit

- 数据：**NumPy + pandas + Polars**
- 可视化：**Matplotlib + Seaborn + Plotly**
- 数学：你能从公式推导**走到 NumPy 实现**
- DL：**PyTorch autograd** 入门
- 工程：venv + git + pre-commit + 可复现训练

### 已能解释的"为什么" / "Why"s you can now answer

- 为什么矩阵 `solve` 比 `inv` 好
- 为什么分类用 cross-entropy 不用 MSE
- 为什么 Adam 在 DL 里好用
- 为什么 SGD 噪声有助逃出鞍点
- 为什么 NN 损失非凸但还能训
- 为什么 lock file 比 requirements.txt 更可信

### 下一站 / What's next

**Part 1 · SQL & Databases** —— 面试第一关。从 SELECT 到窗口函数，再到 dbt 数据仓库设计。
**Part 1 · SQL & Databases** — the first round of every DS interview. From SELECT to window functions to dbt.

---

> 📜 这一节没有可视化、没有数学。但它教的是**让 Senior 和 Junior 拉开差距的真本事**——把代码组织、记录、可复现化的能力。
> No viz, no math here — but the engineering skills that separate Senior from Junior DS.
